# E1: Episodic Trace Memory for Code Generation

Can an LLM generate better code by retrieving **similar past solved problems** as procedural memory?

| Component | Choice |
|-----------|--------|
| **Training corpus** | MBPP (974 Python problems with solutions) |
| **Test benchmark** | HumanEval (164 Python problems, pass@1) |
| **Embedding model** | all-MiniLM-L6-v2 (local, free) |
| **LLM** | gpt-4o-mini via OpenAI API |
| **Memory backend** | CVX with MetadataFilter + episode encoding |

### Approach

Each MBPP solution is stored as a 3-step episode:
1. **Understand**: problem description embedding
2. **Plan**: pseudocode/approach embedding
3. **Solve**: final solution embedding

At test time, embed the HumanEval problem, retrieve the top-k most similar MBPP episodes (filtered by `reward >= 1.0` = passing solutions), and inject them as few-shot examples with full procedural context.

### Conditions

| Condition | Description |
|-----------|-------------|
| **NoMemory** | Zero-shot: just the problem description |
| **RandomFewShot** | 3 random MBPP solutions as examples |
| **SemanticRetrieval** | 3 most similar MBPP by cosine (flat store) |
| **CVX-Episodic** | 3 most similar via CVX causal_search + metadata filter |

In [1]:
import os, json, time, hashlib
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
from openai import OpenAI

# === MODEL CONFIG ===
# Option A: Ollama local (free, needs `ollama serve` running)
# Run on HPC: ollama pull qwen2.5-coder:7b-instruct
USE_OLLAMA = True
OLLAMA_MODEL = 'qwen2.5-coder:7b-instruct'
OLLAMA_URL = 'http://hpc.glaciar.lab:11434/v1'

# Option B: OpenAI API (set OPENAI_API_KEY env var)
# USE_OLLAMA = False
# OPENAI_MODEL = 'gpt-4o-mini'  # ~$0.10 for full run

if USE_OLLAMA:
    client = OpenAI(base_url=OLLAMA_URL, api_key='ollama')
    LLM_MODEL = OLLAMA_MODEL
else:
    assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY'
    client = OpenAI()
    LLM_MODEL = 'gpt-4o-mini'

EMBED_MODEL = 'all-MiniLM-L6-v2'
TOP_K = 3
MAX_EVAL = 50  # subset for speed (164 for full HumanEval)

DATA_DIR = Path('../data/episodic')
DATA_DIR.mkdir(parents=True, exist_ok=True)

embedder = SentenceTransformer(EMBED_MODEL)
D = embedder.get_sentence_embedding_dimension()
print(f'Embedding: {EMBED_MODEL} (D={D})')
print(f'LLM: {LLM_MODEL} ({"Ollama" if USE_OLLAMA else "OpenAI"})')

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9175.18it/s]


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding: all-MiniLM-L6-v2 (D=384)
LLM: qwen2.5-coder:7b-instruct (Ollama)


## 1. Load Datasets

In [2]:
# Download datasets if not cached
from datasets import load_dataset

# MBPP — training corpus
mbpp = load_dataset('google-research-datasets/mbpp', 'sanitized', split='train+test+prompt')
print(f'MBPP: {len(mbpp)} problems')

# HumanEval — test benchmark
humaneval = load_dataset('openai/openai_humaneval', split='test')
print(f'HumanEval: {len(humaneval)} problems')

# Preview
print(f'\nMBPP example:')
print(f'  Task: {mbpp[0]["prompt"][:100]}...')
print(f'  Code: {mbpp[0]["code"][:100]}...')
print(f'\nHumanEval example:')
print(f'  Prompt: {humaneval[0]["prompt"][:100]}...')

MBPP: 384 problems


HumanEval: 164 problems

MBPP example:
  Task: Write a python function to find the first repeated character in a given string....
  Code: def first_repeated_char(str1):
  for index,c in enumerate(str1):
    if str1[:index+1].count(c) > 1:...

HumanEval example:
  Prompt: from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
  ...


## 2. Build Episodic Memory from MBPP

Each MBPP problem becomes a 3-step episode in CVX:
- Step 0: problem description (embed the prompt)
- Step 1: solution approach (embed the code's docstring/first comment)
- Step 2: full solution (embed the complete code)

Metadata includes `reward=1.0` (all MBPP solutions pass their tests), `task_id`, and the actual code text.

In [3]:
import chronos_vector as cvx

INDEX_PATH = str(DATA_DIR / 'mbpp_episodic.cvx')

# Episode encoding: episode_id (48 bits) | step_index (16 bits)
def encode_entity(episode_id, step_index):
    return (episode_id << 16) | step_index

def decode_entity(entity_id):
    return entity_id >> 16, entity_id & 0xFFFF

# Build or load index
if os.path.exists(INDEX_PATH):
    index = cvx.TemporalIndex.load(INDEX_PATH)
    print(f'Loaded cached index: {len(index)} points')
else:
    print('Building episodic memory from MBPP...')
    index = cvx.TemporalIndex(m=16, ef_construction=100, ef_search=50)
    
    episode_metadata = {}  # episode_id -> {prompt, code, task_id}
    
    for ep_idx, problem in enumerate(mbpp):
        prompt = problem['prompt']
        code = problem['code']
        task_id = problem.get('task_id', ep_idx)
        
        # Step 0: problem description
        v_problem = embedder.encode(prompt).tolist()
        eid_0 = encode_entity(ep_idx, 0)
        index.insert(entity_id=eid_0, timestamp=ep_idx * 1000, vector=v_problem)
        
        # Step 1: solution approach (first line of code as plan)
        first_line = code.split('\n')[0] if code else prompt
        v_plan = embedder.encode(f'Plan: {first_line}').tolist()
        eid_1 = encode_entity(ep_idx, 1)
        index.insert(entity_id=eid_1, timestamp=ep_idx * 1000 + 1, vector=v_plan)
        
        # Step 2: full solution
        v_solution = embedder.encode(code[:500]).tolist()
        eid_2 = encode_entity(ep_idx, 2)
        index.insert(entity_id=eid_2, timestamp=ep_idx * 1000 + 2, vector=v_solution)
        
        episode_metadata[ep_idx] = {
            'prompt': prompt,
            'code': code,
            'task_id': task_id,
            'reward': 1.0,
        }
    
    index.save(INDEX_PATH)
    # Save metadata separately
    with open(str(DATA_DIR / 'mbpp_metadata.json'), 'w') as f:
        json.dump(episode_metadata, f)
    
    print(f'Built index: {len(index)} points ({len(mbpp)} episodes × 3 steps)')

# Load metadata
with open(str(DATA_DIR / 'mbpp_metadata.json')) as f:
    episode_metadata = {int(k): v for k, v in json.load(f).items()}
print(f'{len(episode_metadata)} episodes with metadata')

Building episodic memory from MBPP...


Built index: 1152 points (384 episodes × 3 steps)
384 episodes with metadata


## 3. Retrieval Functions

In [4]:
def retrieve_episodes_cvx(query_text, top_k=TOP_K):
    """Retrieve similar MBPP episodes via CVX.
    
    Searches for step_0 (problem description) vectors similar to the query,
    then reconstructs full episodes using timestamp encoding.
    
    Encoding scheme: timestamp = ep_idx * 1000 + step_index
    So ep_idx = timestamp // 1000, step_index = timestamp % 1000
    """
    query_vec = embedder.encode(query_text).tolist()
    
    # Search — will match against all steps; we filter to step_0 by timestamp
    results = index.search(vector=query_vec, k=top_k * 10)  # over-fetch
    
    # Filter to step_0 entries and deduplicate by episode
    seen_episodes = set()
    episodes = []
    for node_id, timestamp, score in results:
        ep_idx = timestamp // 1000
        step_idx = timestamp % 1000
        
        if step_idx != 0:  # only match on problem descriptions
            continue
        if ep_idx in seen_episodes:
            continue
        seen_episodes.add(ep_idx)
        
        if ep_idx in episode_metadata:
            episodes.append({
                'episode_id': ep_idx,
                'similarity': score,
                **episode_metadata[ep_idx],
            })
        
        if len(episodes) >= top_k:
            break
    
    return episodes


def retrieve_random(top_k=TOP_K):
    """Random baseline: pick random MBPP solutions."""
    indices = np.random.choice(len(episode_metadata), size=top_k, replace=False)
    return [episode_metadata[int(i)] for i in indices]


def format_episodes_for_prompt(episodes):
    """Format retrieved episodes as few-shot examples."""
    lines = []
    for i, ep in enumerate(episodes, 1):
        lines.append(f'### Example {i} (from similar past problem)')
        lines.append(f'Problem: {ep["prompt"]}')
        lines.append(f'Solution:')
        lines.append(f'```python')
        lines.append(ep['code'])
        lines.append(f'```')
        lines.append('')
    return '\n'.join(lines)


# Test retrieval
test_query = humaneval[0]['prompt']
episodes = retrieve_episodes_cvx(test_query)
print(f'Query: {test_query[:80]}...')
print(f'\nRetrieved {len(episodes)} episodes:')
for ep in episodes:
    print(f'  [{ep["episode_id"]}] sim={ep["similarity"]:.3f}: {ep["prompt"][:60]}...')

Query: from typing import List


def has_close_elements(numbers: List[float], threshold...

Retrieved 3 episodes:
  [82] sim=0.909: Write a python function to get the difference between two li...
  [133] sim=0.916: Write a python function to find smallest number in a list....
  [77] sim=0.934: Write a python function to find the minimum difference betwe...


## 4. LLM Code Generation

In [5]:
def generate_code(problem_prompt, examples_text='', model=LLM_MODEL):
    """Generate Python code for a HumanEval problem."""
    system = (
        'You are an expert Python programmer. Complete the function based on '
        'the docstring. Return ONLY the function body (no explanation, no markdown). '
        'The function signature is already provided.'
    )
    
    user_parts = []
    if examples_text:
        user_parts.append('Here are similar solved problems for reference:\n')
        user_parts.append(examples_text)
        user_parts.append('\n---\nNow solve this problem:\n')
    user_parts.append(problem_prompt)
    
    response = client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': '\n'.join(user_parts)},
        ],
        temperature=0.0,
        max_tokens=1024,
    )
    return response.choices[0].message.content


def extract_function_body(generated, prompt):
    """Extract the function body from generated code."""
    code = generated.strip()
    # Remove markdown fences if present
    if code.startswith('```'):
        code = code.split('\n', 1)[1] if '\n' in code else code[3:]
    if code.endswith('```'):
        code = code[:-3]
    code = code.strip()
    return code


def evaluate_solution(prompt, generated_body, test_code, entry_point):
    """Run the test cases against the generated solution."""
    full_code = prompt + generated_body + '\n' + test_code
    try:
        exec_globals = {}
        exec(full_code, exec_globals)
        # Run the check function
        check_fn = f'check({entry_point})'
        exec(check_fn, exec_globals)
        return True
    except Exception as e:
        return False


# Quick test
test_gen = generate_code(humaneval[0]['prompt'])
print(f'Generated ({len(test_gen)} chars):\n{test_gen[:200]}...')

Generated (266 chars):
```python
def has_close_elements(numbers: List[float], threshold: float) -> bool:
    for i in range(len(numbers)):
        for j in range(i + 1, len(numbers)):
            if abs(numbers[i] - numbers...


## 5. Run Evaluation

Evaluate each condition on the HumanEval subset.

In [6]:
conditions = {
    'NoMemory': lambda p: '',
    'RandomFewShot': lambda p: format_episodes_for_prompt(retrieve_random()),
    'CVX-Episodic': lambda p: format_episodes_for_prompt(retrieve_episodes_cvx(p)),
}

results = {name: [] for name in conditions}
eval_subset = list(humaneval)[:MAX_EVAL]

for i, problem in enumerate(eval_subset):
    prompt = problem['prompt']
    test_code = problem['test']
    entry_point = problem['entry_point']
    task_id = problem['task_id']
    
    for cond_name, get_context in conditions.items():
        context = get_context(prompt)
        generated = generate_code(prompt, context)
        body = extract_function_body(generated, prompt)
        passed = evaluate_solution(prompt, body, test_code, entry_point)
        results[cond_name].append({
            'task_id': task_id,
            'passed': passed,
            'generated_len': len(body),
        })
    
    if (i + 1) % 10 == 0:
        rates = {name: sum(r['passed'] for r in res) / len(res)
                 for name, res in results.items()}
        print(f'[{i+1}/{MAX_EVAL}] pass@1: {rates}')

print('\n=== FINAL RESULTS ===')
for name, res in results.items():
    passed = sum(r['passed'] for r in res)
    total = len(res)
    print(f'{name:20s}: {passed}/{total} = {passed/total:.1%} pass@1')

[10/50] pass@1: {'NoMemory': 0.7, 'RandomFewShot': 1.0, 'CVX-Episodic': 0.9}


[20/50] pass@1: {'NoMemory': 0.55, 'RandomFewShot': 0.9, 'CVX-Episodic': 0.85}


[30/50] pass@1: {'NoMemory': 0.5, 'RandomFewShot': 0.8, 'CVX-Episodic': 0.8333333333333334}


[40/50] pass@1: {'NoMemory': 0.5, 'RandomFewShot': 0.825, 'CVX-Episodic': 0.85}


[50/50] pass@1: {'NoMemory': 0.52, 'RandomFewShot': 0.82, 'CVX-Episodic': 0.86}

=== FINAL RESULTS ===
NoMemory            : 26/50 = 52.0% pass@1
RandomFewShot       : 41/50 = 82.0% pass@1
CVX-Episodic        : 43/50 = 86.0% pass@1


In [7]:
# Visualize results
import plotly.graph_objects as go

names = list(results.keys())
rates = [sum(r['passed'] for r in results[n]) / len(results[n]) for n in names]

fig = go.Figure(data=go.Bar(
    x=names, y=[r * 100 for r in rates],
    text=[f'{r:.1%}' for r in rates],
    textposition='outside',
    marker_color=['#95a5a6', '#3498db', '#e74c3c'],
))
fig.update_layout(
    title=f'HumanEval pass@1 by Memory Condition (n={MAX_EVAL}, model={LLM_MODEL})',
    yaxis_title='pass@1 (%)', height=400,
    yaxis=dict(range=[0, 100]),
)
fig.show()

## 6. Analysis: When Does Episodic Memory Help?

In [8]:
import pandas as pd

# Build comparison dataframe
df_results = pd.DataFrame([
    {'task_id': r['task_id'], 'condition': cond, 'passed': r['passed']}
    for cond, res in results.items()
    for r in res
])

pivot = df_results.pivot(index='task_id', columns='condition', values='passed')

# Cases where CVX-Episodic helps vs hurts
if 'CVX-Episodic' in pivot.columns and 'NoMemory' in pivot.columns:
    helped = pivot[(pivot['CVX-Episodic'] == True) & (pivot['NoMemory'] == False)]
    hurt = pivot[(pivot['CVX-Episodic'] == False) & (pivot['NoMemory'] == True)]
    both = pivot[(pivot['CVX-Episodic'] == True) & (pivot['NoMemory'] == True)]
    neither = pivot[(pivot['CVX-Episodic'] == False) & (pivot['NoMemory'] == False)]
    
    print(f'CVX-Episodic HELPED (only CVX passed): {len(helped)}')
    print(f'CVX-Episodic HURT  (only NoMemory passed): {len(hurt)}')
    print(f'Both passed: {len(both)}')
    print(f'Neither passed: {len(neither)}')
    print(f'\nNet improvement: {len(helped) - len(hurt):+d} problems')

CVX-Episodic HELPED (only CVX passed): 18
CVX-Episodic HURT  (only NoMemory passed): 1
Both passed: 25
Neither passed: 6

Net improvement: +17 problems


## 7. Retrieval Quality Analysis

In [9]:
# Analyze retrieval quality: how similar are retrieved episodes to test problems?
similarities = []
for problem in eval_subset:
    episodes = retrieve_episodes_cvx(problem['prompt'])
    if episodes:
        similarities.append(episodes[0]['similarity'])

print(f'Top-1 retrieval similarity:')
print(f'  Mean: {np.mean(similarities):.4f}')
print(f'  Median: {np.median(similarities):.4f}')
print(f'  Min: {np.min(similarities):.4f}')
print(f'  Max: {np.max(similarities):.4f}')

fig = go.Figure(data=go.Histogram(x=similarities, nbinsx=30, marker_color='#3498db'))
fig.update_layout(
    title='Distribution of Top-1 Retrieval Similarity (MBPP→HumanEval)',
    xaxis_title='Cosine Similarity', yaxis_title='Count', height=350,
)
fig.show()

Top-1 retrieval similarity:
  Mean: 0.8216
  Median: 0.8674
  Min: 0.4084
  Max: 1.1149


## Summary

In [10]:
print('=== E1: EPISODIC CODING BENCHMARK ===')
print(f'Model: {LLM_MODEL}')
print(f'Training corpus: MBPP ({len(episode_metadata)} episodes)')
print(f'Test: HumanEval (n={MAX_EVAL})')
print(f'Embedding: {EMBED_MODEL} (D={D})')
print()
for name, res in results.items():
    passed = sum(r['passed'] for r in res)
    total = len(res)
    print(f'  {name:20s}: {passed}/{total} = {passed/total:.1%}')
print()
print('CVX features used: insert, search, episode encoding')
print('Key metric: does retrieval-augmented generation outperform zero-shot?')

=== E1: EPISODIC CODING BENCHMARK ===
Model: qwen2.5-coder:7b-instruct
Training corpus: MBPP (384 episodes)
Test: HumanEval (n=50)
Embedding: all-MiniLM-L6-v2 (D=384)

  NoMemory            : 26/50 = 52.0%
  RandomFewShot       : 41/50 = 82.0%
  CVX-Episodic        : 43/50 = 86.0%

CVX features used: insert, search, episode encoding
Key metric: does retrieval-augmented generation outperform zero-shot?
